In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import sys
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier 

In [2]:
df = pd.read_csv(r"C:\Users\K.Swetha\Desktop\dataset.csv")

In [3]:
df

,Age,Gender,Blood Pressure,Cholesterol,Glucose,Smoking,Alcohol Consumption,Exercise,BMI,Family History,Heart Disease,Diabetes,Stroke,Kidney Disease,Cancer,Alzheimer's Disease,COPD,Liver Disease,Parkinson's Disease,Tuberculosis
0,69,Male,High,High,High,Yes,No,No,35.671099,No,1,0,0,0,1,0,0,0,0,0
1,32,Male,Low,High,Normal,Yes,No,Yes,38.554188,Yes,0,1,0,0,0,0,0,1,0,0
2,89,Female,Normal,High,Normal,No,No,Yes,18.932964,Yes,1,0,0,0,0,0,0,0,0,0
3,78,Male,High,High,High,No,No,Yes,21.806350,Yes,0,1,1,0,1,0,0,1,0,0
4,38,Male,Low,Normal,Normal,Yes,Yes,Yes,37.552683,No,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,Female,Low,High,Normal,No,No,No,31.960176,Yes,1,0,0,0,0,0,0,0,0,0
996,51,Female,High,High,Normal,No,Yes,Yes,20.118492,Yes,0,0,0,0,0,0,0,0,0,0
997,72,Female,Normal,High,Normal,Yes,No,No,20.916536,Yes,0,0,0,0,0,0,0,0,0,0
998,49,Male,Normal,High,High,Yes,No,Yes,19.560143,Yes,0,0,0,0,0,1,0,0,0,0


In [4]:
df.columns = df.columns.str.strip()

In [5]:
target_col = "Stroke"

In [6]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [7]:
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    if col != target_col:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    df.drop(columns=[target_col])
    df[target_col]

In [8]:
x = df.drop(columns=[target_col])
y = df[target_col]

In [9]:
if y.dtype == 'object':
    target_le = LabelEncoder()
    y = target_le.fit_transform(y)

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [12]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [13]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "XGBoost": XGBClassifier(n_estimators = 100, max_depth=5, learning_rate=0.05,eval_metric='logloss', random_state=42)
}

In [14]:
best_accuracy = 0
best_model_name = None

In [15]:
print("--- MODEL PERFORMANCE EVALUATION ---")
for name, model in models.items():
    model.fit(x_train_scaled, y_train)
    y_train_pred = model.predict(x_train_scaled)
    y_test_pred = model.predict(x_test_scaled)
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    print(f"\n==== {name} ====")
    print(f"Training Accuracy : {train_acc:.4f}")
    print(f"Testing Accuracy : {test_acc:.4f}")


--- MODEL PERFORMANCE EVALUATION ---

==== Logistic Regression ====
Training Accuracy : 0.8662
Testing Accuracy : 0.8650

==== Random Forest ====
Training Accuracy : 0.9425
Testing Accuracy : 0.8650

==== XGBoost ====
Training Accuracy : 0.8788
Testing Accuracy : 0.8650


In [16]:
#over fitting check logic
gap = train_acc - test_acc
print(f"train - Test Gap : {gap:.4f}")
if gap > 0.12:
    print("Alert : Potential Overfitting detected! Model performs much better on training data.")
elif gap < 0.0:
    print("Note : Underfitting or highly generalized varient.")
else:
    print("Stable : Model generalized effectively without sever overfitting.")
print("\nTest Classsification Report:")
print(classification_report(y_test, y_test_pred))

train - Test Gap : 0.0138
Stable : Model generalized effectively without sever overfitting.

Test Classsification Report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.93       173
           1       0.00      0.00      0.00        27

    accuracy                           0.86       200
   macro avg       0.43      0.50      0.46       200
weighted avg       0.75      0.86      0.80       200



c:\Users\K.Swetha\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\K.Swetha\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\K.Swetha\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [17]:
#Persisit the highest-performing model configuration
if test_acc > best_accuracy:
    best_accuracy = test_acc
    best_model_name = name
    joblib.dump(model, "best_disease_model.pkl")
print(f"\npipeline Saved '{best_model_name}' as 'best_disease_model.pkl' (Accuracy: {best_accuracy:.4f})")


pipeline Saved 'XGBoost' as 'best_disease_model.pkl' (Accuracy: 0.8650)


In [18]:
x_test = df.drop(columns=['Stroke'])
predictions = model.predict(x_test)


In [19]:
x_test = df

In [23]:
x_test = x_test[['Stroke']]

In [25]:
df = pd.read_csv(r"C:\Users\K.Swetha\Desktop\dataset.csv")

In [26]:
df

,Age,Gender,Blood Pressure,Cholesterol,Glucose,Smoking,Alcohol Consumption,Exercise,BMI,Family History,Heart Disease,Diabetes,Stroke,Kidney Disease,Cancer,Alzheimer's Disease,COPD,Liver Disease,Parkinson's Disease,Tuberculosis
0,69,Male,High,High,High,Yes,No,No,35.671099,No,1,0,0,0,1,0,0,0,0,0
1,32,Male,Low,High,Normal,Yes,No,Yes,38.554188,Yes,0,1,0,0,0,0,0,1,0,0
2,89,Female,Normal,High,Normal,No,No,Yes,18.932964,Yes,1,0,0,0,0,0,0,0,0,0
3,78,Male,High,High,High,No,No,Yes,21.806350,Yes,0,1,1,0,1,0,0,1,0,0
4,38,Male,Low,Normal,Normal,Yes,Yes,Yes,37.552683,No,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,Female,Low,High,Normal,No,No,No,31.960176,Yes,1,0,0,0,0,0,0,0,0,0
996,51,Female,High,High,Normal,No,Yes,Yes,20.118492,Yes,0,0,0,0,0,0,0,0,0,0
997,72,Female,Normal,High,Normal,Yes,No,No,20.916536,Yes,0,0,0,0,0,0,0,0,0,0
998,49,Male,Normal,High,High,Yes,No,Yes,19.560143,Yes,0,0,0,0,0,1,0,0,0,0


In [28]:
df.columns = df.columns.str.strip()

In [29]:
x_test = df

In [30]:
df

,Age,Gender,Blood Pressure,Cholesterol,Glucose,Smoking,Alcohol Consumption,Exercise,BMI,Family History,Heart Disease,Diabetes,Stroke,Kidney Disease,Cancer,Alzheimer's Disease,COPD,Liver Disease,Parkinson's Disease,Tuberculosis
0,69,Male,High,High,High,Yes,No,No,35.671099,No,1,0,0,0,1,0,0,0,0,0
1,32,Male,Low,High,Normal,Yes,No,Yes,38.554188,Yes,0,1,0,0,0,0,0,1,0,0
2,89,Female,Normal,High,Normal,No,No,Yes,18.932964,Yes,1,0,0,0,0,0,0,0,0,0
3,78,Male,High,High,High,No,No,Yes,21.806350,Yes,0,1,1,0,1,0,0,1,0,0
4,38,Male,Low,Normal,Normal,Yes,Yes,Yes,37.552683,No,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,Female,Low,High,Normal,No,No,No,31.960176,Yes,1,0,0,0,0,0,0,0,0,0
996,51,Female,High,High,Normal,No,Yes,Yes,20.118492,Yes,0,0,0,0,0,0,0,0,0,0
997,72,Female,Normal,High,Normal,Yes,No,No,20.916536,Yes,0,0,0,0,0,0,0,0,0,0
998,49,Male,Normal,High,High,Yes,No,Yes,19.560143,Yes,0,0,0,0,0,1,0,0,0,0


In [31]:
x_test = df